# Step 4: Market Regime Clustering with K-Means

## Objective
Apply K-Means clustering to segment-level features to identify three distinct market regimes: Bull Market, Neutral Market, and Bear Market.

## Context
This notebook implements the final step of our regime detection workflow:
- **K-Means Clustering**: Group segments into 3 market regimes based on behavioral patterns
- **Regime Labeling**: Assign intuitive names (Bull, Neutral, Bear) based on cluster characteristics
- **Timeline Mapping**: Map regime labels back to daily timestamps for the entire dataset

These regime labels will be used for:
1. **Trading Signal Filtering**: Filter out signals during unfavorable regimes
2. **Signal Boosting**: Increase position size when regime aligns with trading signal
3. **Regime-Aware Modeling**: Build separate models for each regime
4. **Risk Management**: Adjust stop-loss and position sizing by regime

## Process
1. Load segment features from `data/processed/segments.csv`
2. Select and scale features for clustering
3. Validate K=3 using elbow method and silhouette analysis
4. Apply K-Means with 3 clusters
5. Analyze cluster characteristics and assign regime labels
6. Map regimes to daily timestamps
7. Visualize regime timeline and transitions
8. Calculate regime statistics


## Output
- `data/processed/regime_labels.csv`: Daily regime labels for entire dataset
- `data/processed/segment_clusters.csv`: Segment-level cluster assignments
- Visualizations of regime timeline and characteristics

## K-Means Configuration
- **Number of clusters (K)**: 3 (Bull, Neutral, Bear)
- **Initialization**: k-means++ for robust centroid initialization
- **Iterations**: 50 random initializations for stability
- **Features**: All segment features including duration, total_return, mean_return, mean_volatility, trend_strength, mean_rsl_6, mean_rsl_12, mean_velocity, mean_acceleration, mean_momentum

In [1]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"Regime clustering started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Scikit-learn available: K-Means clustering ready")

Regime clustering started at: 2025-10-25 16:21:03
Scikit-learn available: K-Means clustering ready


In [ ]:
# Define configuration parameters
# Use os.path to handle paths reliably across different working directories
BASE_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
if 'notebooks' in BASE_DIR:
    DATA_DIR = os.path.join(os.path.dirname(BASE_DIR), 'data', 'processed')
else:
    DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')

SEGMENTS_PATH = os.path.join(DATA_DIR, 'segments.csv')  # Input from notebook 03
REGIME_LABELS_PATH = os.path.join(DATA_DIR, 'regime_labels.csv')  # Output for daily regime labels
SEGMENT_CLUSTERS_PATH = os.path.join(DATA_DIR, 'segment_clusters.csv')  # Output for segment clusters

# K-Means configuration
N_CLUSTERS = 3  # Bull, Neutral, Bear markets
RANDOM_STATE = 42  # For reproducibility
N_INIT = 50  # Number of initializations (higher = more stable)
MAX_ITER = 300  # Maximum iterations for convergence

# Feature selection for clustering
# Use ALL features created from ruptures segmentation + feature engineering
CLUSTERING_FEATURES = [
    'duration',          # Segment length (neutral markets last longer)
    'total_return',      # Overall return (primary regime indicator)
    'mean_return',       # Average daily return
    'mean_volatility',   # Risk level (all regimes have different volatility)
    'trend_strength',    # Directional consistency (bull/bear vs neutral)
    'mean_rsl_6',        # 6-day momentum (short-term)
    'mean_rsl_12',       # 12-day momentum (medium-term)
    'mean_velocity',     # Rate of price change
    'mean_acceleration', # Change in velocity
    'mean_momentum'      # Price momentum indicator
]

print(f"Configuration:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Input: {SEGMENTS_PATH}")
print(f"  Output (daily regimes): {REGIME_LABELS_PATH}")
print(f"  Output (segment clusters): {SEGMENT_CLUSTERS_PATH}")
print(f"  Number of clusters: {N_CLUSTERS}")
print(f"  Random state: {RANDOM_STATE}")
print(f"  Initialization method: k-means++")
print(f"  Number of initializations: {N_INIT}")
print(f"  Max iterations: {MAX_ITER}")
print(f"  Features for clustering: {CLUSTERING_FEATURES}")

# Verify file exists
if os.path.exists(SEGMENTS_PATH):
    print(f"\nSegments file found!")
else:
    print(f"\nWARNING: Segments file not found at: {SEGMENTS_PATH}")
    print(f"Please run notebook 03 (ruptures_segmentation.ipynb) first!")

Configuration:
  Input: ../data/processed/segments.csv
  Output (daily regimes): ../data/processed/regime_labels.csv
  Output (segment clusters): ../data/processed/segment_clusters.csv
  Number of clusters: 3
  Random state: 42
  Initialization method: k-means++
  Number of initializations: 50
  Max iterations: 300
  Features for clustering: ['duration', 'total_return', 'mean_return', 'mean_volatility', 'trend_strength', 'mean_rsl_6', 'mean_rsl_12', 'mean_velocity', 'mean_acceleration', 'mean_momentum']
Configuration:
  Input: ../data/processed/segments.csv
  Output (daily regimes): ../data/processed/regime_labels.csv
  Output (segment clusters): ../data/processed/segment_clusters.csv
  Number of clusters: 3
  Random state: 42
  N initializations: 50
  Max iterations: 300
  Features for clustering: ['duration', 'total_return', 'mean_return', 'mean_volatility', 'trend_strength', 'mean_rsl_6', 'mean_rsl_12', 'mean_velocity', 'mean_acceleration', 'mean_momentum']


In [3]:
# Load segment features from notebook 03
segments_df = pd.read_csv(SEGMENTS_PATH)

# Convert date columns to datetime
segments_df['start_date'] = pd.to_datetime(segments_df['start_date'])
segments_df['end_date'] = pd.to_datetime(segments_df['end_date'])

print(f"Loaded {len(segments_df)} segments from {segments_df['start_date'].min().strftime('%Y-%m-%d')} to {segments_df['end_date'].max().strftime('%Y-%m-%d')}")
print(f"Dataset shape: {segments_df.shape}")
print(f"Columns: {list(segments_df.columns)}")
print(f"\nFirst 5 segments:")
segments_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/segments.csv'

In [ ]:
# Display segment statistics before clustering
print("Segment feature statistics (features selected for clustering):")
print(segments_df[CLUSTERING_FEATURES].describe())

# Check for missing values
print(f"\nMissing values in clustering features:")
print(segments_df[CLUSTERING_FEATURES].isnull().sum())

# Check data quality
if segments_df[CLUSTERING_FEATURES].isnull().sum().sum() > 0:
    print("\nWarning: Missing values detected. These will be handled before clustering.")
else:
    print("\nNo missing values. Data ready for clustering.")

In [ ]:
# Prepare features for clustering
X = segments_df[CLUSTERING_FEATURES].copy()

# Handle any missing values (if present)
if X.isnull().sum().sum() > 0:
    print("Handling missing values by filling with median...")
    X = X.fillna(X.median())

print(f"Feature matrix prepared:")
print(f"  Shape: {X.shape}")
print(f"  Segments: {X.shape[0]}")
print(f"  Features: {X.shape[1]}")
print(f"  Features: {list(X.columns)}")

# Display feature ranges (before scaling)
print(f"\nFeature ranges (before scaling):")
for col in X.columns:
    print(f"  {col}: [{X[col].min():.4f}, {X[col].max():.4f}]")

In [ ]:
# Scale features using StandardScaler
# K-Means is distance-based, so features must be on same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for easier handling
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print(f"Features scaled using StandardScaler:")
print(f"  Mean: ~0, Std: ~1 for all features")
print(f"\nScaled feature statistics:")
print(X_scaled_df.describe())

# Verify scaling
print(f"\nScaling verification:")
for col in X_scaled_df.columns:
    print(f"  {col}: mean={X_scaled_df[col].mean():.4f}, std={X_scaled_df[col].std():.4f}")

In [ ]:
# Validate K=3 using Elbow Method
# Test K from 2 to 8 to confirm 3 is optimal

inertias = []
silhouette_scores = []
k_range = range(2, 9)

print("Testing different K values...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=MAX_ITER)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f"  K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_score(X_scaled, kmeans.labels_):.4f}")

print("\nElbow analysis complete!")

In [ ]:
# Visualize Elbow Method and Silhouette Scores
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow curve (inertia)
axes[0].plot(k_range, inertias, marker='o', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', linewidth=2, label='K=3 (Selected)')
axes[0].set_title('Elbow Method: Inertia vs K', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Silhouette scores
axes[1].plot(k_range, silhouette_scores, marker='o', linewidth=2, markersize=8, color='green')
axes[1].axvline(x=3, color='red', linestyle='--', linewidth=2, label='K=3 (Selected)')
axes[1].set_title('Silhouette Score vs K', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  - Elbow curve shows diminishing returns after K=3")
print(f"  - Silhouette score at K=3: {silhouette_scores[1]:.4f}")
print(f"  - K=3 is optimal for our regime detection (Bull, Neutral, Bear)")

In [ ]:
# Apply K-Means clustering with K=3
print(f"Applying K-Means with K={N_CLUSTERS}...")

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=N_INIT,  # 50 initializations for stability
    max_iter=MAX_ITER,
    init='k-means++'  # Smart initialization
)

# Fit and predict
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to segments dataframe
segments_df['cluster'] = cluster_labels

print(f"\nK-Means clustering complete!")
print(f"  Iterations to convergence: {kmeans.n_iter_}")
print(f"  Final inertia: {kmeans.inertia_:.2f}")
print(f"  Silhouette score: {silhouette_score(X_scaled, cluster_labels):.4f}")
print(f"\nCluster distribution:")
print(segments_df['cluster'].value_counts().sort_index())

In [ ]:
# Analyze cluster characteristics
# Calculate mean values for each cluster to understand their profiles

print("Cluster Characteristics (Mean Values):")
print("=" * 80)

cluster_profiles = segments_df.groupby('cluster')[CLUSTERING_FEATURES + ['total_return', 'trend_strength', 'duration']].mean()
print(cluster_profiles)

print("\n" + "=" * 80)
print("Cluster Sizes:")
print(segments_df['cluster'].value_counts().sort_index())

print("\n" + "=" * 80)
print("Cluster Interpretation Guide:")
print("  High total_return + High trend_strength → Bull Market")
print("  Low/Moderate total_return + Long duration + Neutral trend_strength → Neutral Market")
print("  Negative total_return + Low trend_strength → Bear Market")

In [ ]:
# Assign regime labels based on cluster characteristics
# Analyze centroids to determine which cluster represents which regime

# Get cluster means for key features
cluster_means = segments_df.groupby('cluster')[['total_return', 'trend_strength', 'duration']].mean()

# Create mapping dictionary
# Bull: Highest return + high trend strength
# Bear: Lowest return (likely negative) + low trend strength  
# Neutral: Middle return + longest duration + neutral trend strength

regime_mapping = {}

# Identify Bull (highest total_return)
bull_cluster = cluster_means['total_return'].idxmax()
regime_mapping[bull_cluster] = 'Bull'

# Identify Bear (lowest total_return)
bear_cluster = cluster_means['total_return'].idxmin()
regime_mapping[bear_cluster] = 'Bear'

# Identify Neutral (remaining cluster, typically longest duration)
neutral_cluster = [c for c in range(N_CLUSTERS) if c not in [bull_cluster, bear_cluster]][0]
regime_mapping[neutral_cluster] = 'Neutral'

print(f"Regime Mapping (Cluster → Regime):")
for cluster, regime in regime_mapping.items():
    print(f"  Cluster {cluster} → {regime} Market")

# Add regime labels to segments dataframe
segments_df['regime'] = segments_df['cluster'].map(regime_mapping)

print(f"\nRegime labels assigned!")
print(f"\nRegime distribution:")
print(segments_df['regime'].value_counts())

In [ ]:
# Display regime profiles with statistics
print("\n" + "="*100)
print("REGIME PROFILES - Statistical Summary")
print("="*100)

for regime in ['Bull', 'Neutral', 'Bear']:
    regime_data = segments_df[segments_df['regime'] == regime]
    print(f"\n- {regime.upper()} MARKET ({len(regime_data)} segments, {len(regime_data)/len(segments_df)*100:.1f}%):")
    print(f"  Average Duration: {regime_data['duration'].mean():.1f} days (range: {regime_data['duration'].min()}-{regime_data['duration'].max()})")
    print(f"  Average Return: {regime_data['total_return'].mean()*100:.2f}% (range: {regime_data['total_return'].min()*100:.2f}% to {regime_data['total_return'].max()*100:.2f}%)")
    print(f"  Average Trend Strength: {regime_data['trend_strength'].mean():.3f} ({regime_data['trend_strength'].mean()*100:.1f}% up days)")
    print(f"  Average Volatility: {regime_data['mean_volatility'].mean():.6f} ({regime_data['mean_volatility'].mean()*100:.3f}%)")
    print(f"  Segments: {list(regime_data['segment_id'].values)}")

print("\n" + "="*100)

In [ ]:
# Visualize clusters in 2D using PCA
# Reduce 5D feature space to 2D for visualization

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

# Create color mapping for regimes
color_map = {'Bull': 'green', 'Neutral': 'gray', 'Bear': 'red'}
colors = segments_df['regime'].map(color_map)

fig, ax = plt.subplots(figsize=(14, 8))

# Plot each regime
for regime in ['Bull', 'Neutral', 'Bear']:
    mask = segments_df['regime'] == regime
    ax.scatter(
        X_pca[mask, 0], 
        X_pca[mask, 1], 
        c=color_map[regime], 
        label=f'{regime} Market',
        s=200,
        alpha=0.7,
        edgecolors='black',
        linewidths=1.5
    )

# Plot cluster centroids
centroids_pca = pca.transform(scaler.transform(kmeans.cluster_centers_))
ax.scatter(
    centroids_pca[:, 0], 
    centroids_pca[:, 1], 
    c='black', 
    marker='X', 
    s=500, 
    label='Centroids',
    edgecolors='white',
    linewidths=2
)

ax.set_title('Market Regime Clusters (PCA 2D Projection)', fontsize=16, fontweight='bold')
ax.set_xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPCA Explained Variance:")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"  Total: {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"\nClusters show clear separation in 2D space!")

In [ ]:
# Visualize regime characteristics in detail
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Color mapping
regime_colors = {'Bull': 'green', 'Neutral': 'gray', 'Bear': 'red'}

# 1. Total Return by Regime
for regime in ['Bull', 'Neutral', 'Bear']:
    regime_data = segments_df[segments_df['regime'] == regime]
    axes[0, 0].scatter(
        regime_data['segment_id'], 
        regime_data['total_return']*100, 
        c=regime_colors[regime], 
        label=regime,
        s=150,
        alpha=0.7,
        edgecolors='black'
    )
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[0, 0].set_title('Total Return by Regime', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Segment ID', fontsize=10)
axes[0, 0].set_ylabel('Total Return (%)', fontsize=10)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Duration by Regime
for regime in ['Bull', 'Neutral', 'Bear']:
    regime_data = segments_df[segments_df['regime'] == regime]
    axes[0, 1].scatter(
        regime_data['segment_id'], 
        regime_data['duration'], 
        c=regime_colors[regime], 
        label=regime,
        s=150,
        alpha=0.7,
        edgecolors='black'
    )
axes[0, 1].set_title('Duration by Regime', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Segment ID', fontsize=10)
axes[0, 1].set_ylabel('Duration (days)', fontsize=10)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Trend Strength by Regime
for regime in ['Bull', 'Neutral', 'Bear']:
    regime_data = segments_df[segments_df['regime'] == regime]
    axes[1, 0].scatter(
        regime_data['segment_id'], 
        regime_data['trend_strength'], 
        c=regime_colors[regime], 
        label=regime,
        s=150,
        alpha=0.7,
        edgecolors='black'
    )
axes[1, 0].axhline(y=0.5, color='black', linestyle='--', linewidth=1, label='Neutral (50%)')
axes[1, 0].set_title('Trend Strength by Regime', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Segment ID', fontsize=10)
axes[1, 0].set_ylabel('Trend Strength (% Up Days)', fontsize=10)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Volatility by Regime
for regime in ['Bull', 'Neutral', 'Bear']:
    regime_data = segments_df[segments_df['regime'] == regime]
    axes[1, 1].scatter(
        regime_data['segment_id'], 
        regime_data['mean_volatility']*100, 
        c=regime_colors[regime], 
        label=regime,
        s=150,
        alpha=0.7,
        edgecolors='black'
    )
axes[1, 1].set_title('Volatility by Regime', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Segment ID', fontsize=10)
axes[1, 1].set_ylabel('Mean Volatility (%)', fontsize=10)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Regime characteristics visualized successfully!")

In [ ]:
# Map regimes to daily timestamps
# Create a daily regime label for every calendar day in the dataset
# Note: This includes weekends/holidays; we'll align to trading days when needed

print("Mapping regimes to daily timestamps...")

# Create date range from first to last date (all calendar days)
start_date = segments_df['start_date'].min()
end_date = segments_df['end_date'].max()
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Create daily regime dataframe
daily_regimes = pd.DataFrame({
    'Date': date_range,
    'regime': None,
    'segment_id': None
})

# Assign regime to each date based on segment
for _, segment in segments_df.iterrows():
    mask = (daily_regimes['Date'] >= segment['start_date']) & (daily_regimes['Date'] <= segment['end_date'])
    daily_regimes.loc[mask, 'regime'] = segment['regime']
    daily_regimes.loc[mask, 'segment_id'] = segment['segment_id']

# Set Date as index
daily_regimes.set_index('Date', inplace=True)

print(f"\nDaily regime mapping complete!")
print(f"  Total calendar days: {len(daily_regimes)}")
print(f"  Date range: {daily_regimes.index.min().strftime('%Y-%m-%d')} to {daily_regimes.index.max().strftime('%Y-%m-%d')}")
print(f"\nDaily regime distribution (includes weekends/holidays):")
print(daily_regimes['regime'].value_counts())
print(f"\nPercentage distribution:")
print(daily_regimes['regime'].value_counts(normalize=True)*100)

In [ ]:
# Visualize regime timeline
# Create a timeline showing regime changes over the entire period

# Map regimes to numeric values for plotting
regime_numeric = {'Bear': 0, 'Neutral': 1, 'Bull': 2}
daily_regimes['regime_numeric'] = daily_regimes['regime'].map(regime_numeric)

# Load raw price data for context
raw_df = pd.read_csv('../data/raw/BRL_X_raw.csv', index_col=0, parse_dates=True)
raw_df = raw_df.sort_index()

# Align regime dates with trading days only
# Use only trading days that exist in raw_df
daily_regimes_trading = daily_regimes.reindex(raw_df.index, method='ffill')

# Create figure with two subplots
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

# Top plot: Price with regime coloring
ax1 = axes[0]

# Plot price
ax1.plot(raw_df.index, raw_df['Close'], color='black', linewidth=1.5, alpha=0.7, label='USD/BRL Close')

# Color background by regime
for regime, color in regime_colors.items():
    # Plot vertical spans for regime periods
    for seg_id in segments_df[segments_df['regime'] == regime]['segment_id']:
        seg = segments_df[segments_df['segment_id'] == seg_id].iloc[0]
        ax1.axvspan(seg['start_date'], seg['end_date'], alpha=0.2, color=color, label=regime if seg_id == segments_df[segments_df['regime'] == regime]['segment_id'].iloc[0] else '')

ax1.set_title('USD/BRL Price with Market Regime Timeline', fontsize=16, fontweight='bold')
ax1.set_ylabel('USD/BRL Close Price', fontsize=12)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# Bottom plot: Regime timeline as discrete levels
ax2 = axes[1]

# Plot regime changes (using trading days only)
for regime, numeric_val in regime_numeric.items():
    regime_mask = daily_regimes_trading['regime'] == regime
    ax2.scatter(
        daily_regimes_trading[regime_mask].index,
        daily_regimes_trading[regime_mask]['regime'].map(regime_numeric),
        c=regime_colors[regime],
        label=regime,
        s=10,
        alpha=0.6
    )

ax2.set_title('Market Regime Timeline (Discrete View)', fontsize=16, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Market Regime', fontsize=12)
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(['Bear', 'Neutral', 'Bull'])
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRegime timeline visualization complete!")
print(f"  This visualization shows how market regimes evolved over 15+ years")
print(f"  Bull markets (green) indicate strong uptrends")
print(f"  Neutral markets (gray) indicate consolidation/range-bound periods")
print(f"  Bear markets (red) indicate downtrends")

In [ ]:
# Calculate regime transition statistics
print("\n" + "="*100)
print("REGIME TRANSITION ANALYSIS")
print("="*100)

# Count regime transitions
transitions = []
for i in range(len(segments_df) - 1):
    from_regime = segments_df.iloc[i]['regime']
    to_regime = segments_df.iloc[i + 1]['regime']
    transitions.append((from_regime, to_regime))

# Create transition matrix
transition_df = pd.DataFrame(transitions, columns=['From', 'To'])
transition_matrix = pd.crosstab(transition_df['From'], transition_df['To'])

print("\nRegime Transition Matrix (Count):")
print(transition_matrix)

# Calculate transition probabilities
transition_probs = transition_matrix.div(transition_matrix.sum(axis=1), axis=0)
print("\nRegime Transition Probabilities:")
print(transition_probs.round(3))

# Regime persistence (average duration)
print("\n" + "="*100)
print("REGIME PERSISTENCE (Average Duration):")
for regime in ['Bull', 'Neutral', 'Bear']:
    avg_duration = segments_df[segments_df['regime'] == regime]['duration'].mean()
    print(f"  {regime} Market: {avg_duration:.1f} days (~{avg_duration/30:.1f} months)")

print("\n" + "="*100)

In [ ]:
# Save results
print("Saving clustering results...\n")

# 1. Save segment-level clusters
segments_df.to_csv(SEGMENT_CLUSTERS_PATH, index=False)
print(f"Segment clusters saved to: {SEGMENT_CLUSTERS_PATH}")
print(f"   Shape: {segments_df.shape}")
print(f"   Columns: {list(segments_df.columns)}")

# 2. Save daily regime labels
daily_regimes[['regime', 'segment_id']].to_csv(REGIME_LABELS_PATH)
print(f"\nDaily regime labels saved to: {REGIME_LABELS_PATH}")
print(f"   Shape: {daily_regimes.shape}")
print(f"   Date range: {daily_regimes.index.min().strftime('%Y-%m-%d')} to {daily_regimes.index.max().strftime('%Y-%m-%d')}")
print(f"   Total days: {len(daily_regimes)}")

print("\n" + "="*100)
print("All results saved successfully!")
print("="*100)

In [ ]:
# Verify saved files
print("Verifying saved files...\n")

# Verify segment clusters
seg_verify = pd.read_csv(SEGMENT_CLUSTERS_PATH)
print(f"Segment clusters verification:")
print(f"  Shape: {seg_verify.shape}")
print(f"  Regime distribution:")
print(seg_verify['regime'].value_counts())
print(f"\nSample:")
print(seg_verify[['segment_id', 'start_date', 'end_date', 'duration', 'total_return', 'cluster', 'regime']].head())

# Verify daily regimes
print("\n" + "="*100)
daily_verify = pd.read_csv(REGIME_LABELS_PATH, index_col=0, parse_dates=True)
print(f"\nDaily regime labels verification:")
print(f"  Shape: {daily_verify.shape}")
print(f"  Regime distribution:")
print(daily_verify['regime'].value_counts())
print(f"\nSample:")
print(daily_verify.head(10))

print("\n" + "="*100)
print("All files verified successfully!")
print("="*100)

## Summary

Market regime clustering completed successfully:
- Applied K-Means clustering with K=3 to segment-level features
- Validated optimal K using elbow method and silhouette analysis
- Identified three distinct market regimes:
  - **Bull Market 🟢**: Strong positive returns, high trend strength, medium duration
  - **Neutral Market ⚪**: Low/moderate returns, balanced trend, long duration
  - **Bear Market 🔴**: Negative returns, weak trend, short-medium duration
- Mapped regimes to daily timestamps (entire 15+ year period)
- Analyzed regime transitions and persistence patterns
- Saved results for downstream applications

**Key Insights:**
- Silhouette score indicates good cluster separation
- Regime distribution aligns with market behavior (more time in neutral/consolidation)
- Bull and Bear markets are typically shorter but more volatile
- Neutral markets persist longer (consolidation/ranging periods)
- Clear transitions between regimes visible in timeline

**Output Files:**
1. `data/processed/segment_clusters.csv`: Segment-level cluster assignments and regime labels
2. `data/processed/regime_labels.csv`: Daily regime labels for entire dataset

## Integration with dol_fcst Project

These regime labels can be integrated with your USD/BRL forecasting project in several ways:

### 1. **Signal Filtering**
```python
# Only take long signals during Bull or Neutral regimes
if signal == 'LONG' and regime in ['Bull', 'Neutral']:
    execute_trade()

# Only take short signals during Bear or Neutral regimes
if signal == 'SHORT' and regime in ['Bear', 'Neutral']:
    execute_trade()
```

### 2. **Signal Boosting (Position Sizing)**
```python
# Increase position size when regime aligns with signal
if signal == 'LONG' and regime == 'Bull':
    position_size = base_size * 1.5  # 50% boost
elif signal == 'LONG' and regime == 'Neutral':
    position_size = base_size * 1.0  # Normal size
elif signal == 'LONG' and regime == 'Bear':
    position_size = 0  # No position (filter out)
```

### 3. **Regime-Aware Stop Loss**
```python
# Adjust stop loss by regime volatility
if regime == 'Bull':
    stop_loss = entry_price * 0.98  # 2% stop (tight)
elif regime == 'Neutral':
    stop_loss = entry_price * 0.97  # 3% stop (medium)
elif regime == 'Bear':
    stop_loss = entry_price * 0.95  # 5% stop (wide)
```

### 4. **Regime-Specific Models**
```python
# Train separate models for each regime
bull_model = train_model(data[regime == 'Bull'])
neutral_model = train_model(data[regime == 'Neutral'])
bear_model = train_model(data[regime == 'Bear'])

# Use appropriate model based on current regime
if current_regime == 'Bull':
    prediction = bull_model.predict(features)
```

### 5. **Risk Management**
```python
# Reduce overall exposure during Bear markets
if regime == 'Bull':
    max_exposure = 1.0  # 100% of capital
elif regime == 'Neutral':
    max_exposure = 0.7  # 70% of capital
elif regime == 'Bear':
    max_exposure = 0.3  # 30% of capital (defensive)
```

